In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

from cpcv_analysis.config import XGB_PARAMS, ASSETS, TIMELINES, SYNTHETIC_SCENARIOS, TIMELINE_B
from cpcv_analysis.experiment import run_experiment, _run_experiment_from_arrays
from cpcv_analysis.synthetic import generate_synthetic_prices
from cpcv_analysis.data import build_features

clf_template = XGBClassifier(**XGB_PARAMS)
METHODS = ["cpcv", "wf", "kfold"]
ASSET_TICKERS = list(ASSETS.keys())

os.makedirs("plots", exist_ok=True)
os.makedirs("results", exist_ok=True)
print(f"Assets: {ASSET_TICKERS}")
print(f"Timelines: {[t['name'] for t in TIMELINES]}")
print(f"Methods: {METHODS}")

In [ ]:
real_rows = []

for timeline_cfg in TIMELINES:
    for ticker in ASSET_TICKERS:
        for method in METHODS:
            print(f"Running: {timeline_cfg['name']} | {ticker} | {method}...")
            try:
                metrics, fig = run_experiment(ticker, timeline_cfg, clf_template, method=method)
                fname = f"plots/real_{timeline_cfg['name']}_{ticker}_{method}.png"
                fig.savefig(fname, dpi=100, bbox_inches="tight")
                plt.close(fig)
                row = dict(timeline=timeline_cfg["name"], asset=ticker, method=method, **metrics)
                real_rows.append(row)
                print(f"  -> delta_median={metrics['delta_median']:.3f}, coverage_90={metrics['coverage_90']}")
            except Exception as e:
                print(f"  ERROR: {e}")
                import traceback; traceback.print_exc()

real_df = pd.DataFrame(real_rows)
real_df.to_csv("results/real_experiments.csv", index=False)
print(f"\nCompleted {len(real_rows)} real experiments")
real_df

In [ ]:
display_cols = ["timeline", "asset", "method",
                "delta_median", "coverage_90", "rank_pct",
                "dispersion", "delta_maxDD", "pct_positive"]
real_df[display_cols].round(4)

In [ ]:
synth_rows = []

for scenario in SYNTHETIC_SCENARIOS:
    seed = int(scenario["id"])
    prices_synth = generate_synthetic_prices(scenario, seed=seed)
    X_s, y_s, t1_s, _, fwd_ret_s = build_features(prices_synth)

    timeline_synth = dict(
        name=f"synth_{scenario['id']}",
        wf_start=TIMELINE_B["wf_start"],
        dev_start=TIMELINE_B["dev_start"],
        dev_end=TIMELINE_B["dev_end"],
        retrain_start=TIMELINE_B["retrain_start"],
        retrain_end=TIMELINE_B["retrain_end"],
        holdout_start=TIMELINE_B["holdout_start"],
        holdout_end=TIMELINE_B["holdout_end"],
        download_start=TIMELINE_B["download_start"],
        download_end=TIMELINE_B["download_end"],
    )

    for method in METHODS:
        print(f"Synthetic {scenario['id']} {scenario['name']} | {method}...")
        try:
            metrics, fig = _run_experiment_from_arrays(
                X_s, y_s, t1_s, prices_synth, fwd_ret_s,
                timeline_synth, clf_template, method=method)
            fname = f"plots/synth_{scenario['id']}_{scenario['name']}_{method}.png"
            fig.savefig(fname, dpi=100, bbox_inches="tight")
            plt.close(fig)
            row = dict(scenario_id=scenario["id"], scenario_name=scenario["name"], method=method, **metrics)
            synth_rows.append(row)
            print(f"  -> delta_median={metrics['delta_median']:.3f}")
        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback; traceback.print_exc()

synth_df = pd.DataFrame(synth_rows)
synth_df.to_csv("results/synthetic_experiments.csv", index=False)
print(f"\nCompleted {len(synth_rows)} synthetic experiments")

In [ ]:
display_cols2 = ["scenario_id", "scenario_name", "method",
                 "delta_median", "coverage_90", "rank_pct",
                 "dispersion", "delta_maxDD", "pct_positive"]
synth_df[display_cols2].round(4)

In [ ]:
agg_cols = ["delta_median", "coverage_90", "rank_pct", "dispersion", "delta_maxDD", "pct_positive"]
summary_df = synth_df.groupby("method")[agg_cols].mean().round(4)
summary_df.index.name = "method"
summary_df.to_csv("results/summary_by_method.csv")

print("=== TABLE 3: Average calibration metrics per method (60 synthetic runs) ===")
display(summary_df)

In [ ]:
print(f"Real experiments: {len(real_df)} rows (expected 24)")
print(f"Synthetic experiments: {len(synth_df)} rows (expected 60)")
import glob
n_plots = len(glob.glob("plots/*.png"))
print(f"Figures saved: {n_plots} (expected 84)")
print(f"CSVs in results/: {os.listdir('results/')}")

assert len(real_df) == 24, f"Expected 24 real rows, got {len(real_df)}"
assert len(synth_df) == 60, f"Expected 60 synthetic rows, got {len(synth_df)}"
print("All assertions passed.")